# Audit Logs — Unity Catalog System Tables

**Doel:** Tonen wie wat heeft gedaan met de demo-objecten (catalog, schema's, control table, volumes, pipelines) via de Unity Catalog system audit logs.

## Vereisten

> **Belangrijk:** Dit notebook vereist dat **System Tables** zijn ingeschakeld op het metastore-niveau.  
> Zonder dit zijn de tabellen in `system.access` niet beschikbaar en mislukt het notebook met een duidelijke foutmelding.
>
> **Inschakelen:** Ga naar *Account Console → Metastore → System schemas* en schakel `access` in.  
> Zie ook: [Databricks docs — Monitor usage with system tables](https://docs.databricks.com/en/admin/system-tables/index.html)

## Inhoud

| Sectie | Beschrijving |
|---|---|
| 1 | Preflight-check — `system.access.audit` bereikbaar? |
| 2 | Recente activiteit op de demo-catalog (`DEMO_DEV.*`) |
| 3 | Wie heeft de control table aangemaakt of gewijzigd? |
| 4 | Pipeline run events (Workflows, DLT, Lakeflow Connect) |
| 5 | Compliance & governance narratief |

## Governance use case

De Unity Catalog audit logs geven organisaties een **onveranderlijk spoor** van wie welke data heeft benaderd, welke objecten zijn aangemaakt of gewijzigd, en welke pipelines zijn gestart. Dit is een kernvereiste voor:

- **SOC 2 / ISO 27001** — Aantoonbare toegangscontrole en audittrail
- **AVG / GDPR** — Wie heeft persoonsgegevens ingezien of verwerkt?
- **Interne data governance** — Toezicht op data-wijzigingen en toegangspatronen

De `system.access.audit`-tabel wordt automatisch gevuld door het Databricks-platform en is niet aanpasbaar door gebruikers — waardoor de integriteit van het spoor gewaarborgd is.

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO_DEV", "Demo catalog")
dbutils.widgets.text("lookback_days", "30", "Terugkijkperiode (dagen)")

catalog      = dbutils.widgets.get("catalog")
lookback_days = int(dbutils.widgets.get("lookback_days"))

print(f"Demo catalog     : {catalog}")
print(f"Terugkijkperiode : {lookback_days} dag(en)")
print(f"Audit log bron   : system.access.audit")

## Stap 1 — Preflight-check: `system.access.audit` bereikbaar?

Voordat we queries uitvoeren, controleren we of de `system.access.audit`-tabel beschikbaar is.  
Als de System Tables niet zijn ingeschakeld, geeft het notebook een duidelijke instructie  
om dit te activeren — en stopt het met een `RuntimeError` zodat de gebruiker niet met verwarrende SQL-fouten wordt geconfronteerd.

In [ ]:
# Preflight-check: bestaat system.access.audit?
try:
    spark.sql("DESCRIBE TABLE system.access.audit").collect()
    print("OK — system.access.audit is bereikbaar.")
except Exception as e:
    raise RuntimeError(
        "\n"
        "=========================================================\n"
        "FOUT: system.access.audit is NIET beschikbaar.\n"
        "=========================================================\n"
        "\n"
        "De Unity Catalog System Tables zijn niet ingeschakeld voor dit metastore.\n"
        "\n"
        "Stappen om dit op te lossen:\n"
        "  1. Ga naar Account Console (https://accounts.azuredatabricks.net).\n"
        "  2. Navigeer naar: Catalog → <metastore-naam> → System schemas.\n"
        "  3. Schakel het schema 'access' in.\n"
        "  4. Wacht 10–30 minuten tot de eerste auditrecords verschijnen.\n"
        "  5. Voer dit notebook opnieuw uit.\n"
        "\n"
        "Documentatie: https://docs.databricks.com/en/admin/system-tables/index.html\n"
        f"\nOriginele fout: {e}"
    )

## Stap 2 — Recente activiteit op de demo-catalog

### Wat laat dit zien?

De `system.access.audit`-tabel registreert **elke actie** die een gebruiker of service principal uitvoert op Unity Catalog-objecten. Door te filteren op de naam van de demo-catalog (`DEMO_DEV`) zien we:

- Wie heeft tabellen aangemaakt, gelezen of gewijzigd?
- Welke service principals (Jobs, DLT-pipelines) hebben data weggeschreven?
- Op welke tijdstippen was er activiteit?

**Wat te zoeken in de resultaten:**
- De kolom `user_identity.email` toont de menselijke gebruiker of service principal.
- De kolom `action_name` toont de uitgevoerde operatie (bijv. `createTable`, `createSchema`, `getTable`).
- De kolom `request_params` bevat detail over het specifieke object (catalog, schema, tabelnaam).
- Een hoge frequentie van `getTable`-acties kan wijzen op intensief gebruik of monitoring.

In [ ]:
recent_activity_sql = f"""
SELECT
    DATE(event_time)                            AS event_date,
    event_time,
    user_identity.email                         AS user_or_principal,
    action_name,
    request_params.catalog_name                 AS catalog_name,
    request_params.schema_name                  AS schema_name,
    request_params.table_name                   AS table_name,
    response.status_code                        AS status_code,
    source_ip_address
FROM system.access.audit
WHERE event_time >= CURRENT_TIMESTAMP() - INTERVAL {lookback_days} DAYS
  AND (
        LOWER(request_params.catalog_name) = LOWER('{catalog}')
     OR LOWER(request_params.full_name_arg) LIKE LOWER('{catalog}.%')
  )
ORDER BY event_time DESC
LIMIT 100
"""

recent_df = spark.sql(recent_activity_sql)
count = recent_df.count()
print(f"Recente activiteiten op '{catalog}': {count} records (max 100 getoond)")
recent_df.show(truncate=False)

## Stap 3 — Wie heeft de control table aangemaakt of gewijzigd?

### Wat laat dit zien?

De control table `CONFIG.pipeline_sources` is het hart van de metadata-gedreven pipeline.  
Elke wijziging aan deze tabel heeft direct invloed op het gedrag van de ingest-pipeline.  
Het is dus cruciaal te weten **wie** deze tabel heeft aangemaakt, welke rijen zijn toegevoegd,  
en — als het fout gaat — wie het laatste heeft gewijzigd.

**Wat te zoeken in de resultaten:**
- `createTable` — initiële aanmaak door het bootstrap-notebook (service principal of gebruiker).
- `updateTable` / `writeTable` — wijzigingen (bijv. `is_active = false` of `load_type` aanpassen).
- `deleteTable` — verwijdering (zou alarmbellen moeten doen rinkelen!).
- Vergelijk het tijdstip van wijzigingen met Delta Time Travel-versies (`DESCRIBE HISTORY`) voor volledig traceerbaarheid.

### Combinatie met Delta Time Travel

De audit log vertelt **wie** iets heeft gedaan en **wanneer**. Delta Time Travel  
(zie `demo_showcase/delta_time_travel.ipynb`) vertelt **wat** er is veranderd.  
Samen geven ze een volledig compliance-spoor.

In [ ]:
control_table_audit_sql = f"""
SELECT
    event_time,
    user_identity.email                         AS user_or_principal,
    action_name,
    request_params.catalog_name                 AS catalog_name,
    request_params.schema_name                  AS schema_name,
    request_params.table_name                   AS table_name,
    request_params.full_name_arg                AS full_name_arg,
    response.status_code                        AS status_code,
    request_params                              AS raw_request_params
FROM system.access.audit
WHERE event_time >= CURRENT_TIMESTAMP() - INTERVAL {lookback_days} DAYS
  AND (
        (
          LOWER(request_params.catalog_name) = LOWER('{catalog}')
          AND LOWER(request_params.schema_name) = 'config'
          AND LOWER(request_params.table_name)  = 'pipeline_sources'
        )
     OR LOWER(request_params.full_name_arg) = LOWER('{catalog}.config.pipeline_sources')
  )
ORDER BY event_time DESC
LIMIT 50
"""

control_df = spark.sql(control_table_audit_sql)
count = control_df.count()
print(f"Audit-records voor control table '{catalog}.CONFIG.pipeline_sources': {count}")
control_df.show(truncate=False)

## Stap 4 — Pipeline run events (Workflows, DLT, Lakeflow Connect)

### Wat laat dit zien?

Databricks-pipelines (Jobs, DLT-pipelines, Lakeflow Connect) worden ook geregistreerd in de audit logs via `action_name`-waarden zoals:

| Action | Wat het betekent |
|---|---|
| `runStart` / `runSucceeded` / `runFailed` | Job/Workflow-run gestart of afgerond |
| `pipelineStarted` / `pipelineStopped` | DLT-pipeline gestart of gestopt |
| `createPipeline` / `updatePipeline` | DLT-pipeline definitie aangemaakt of gewijzigd |
| `runCommand` | Notebook-commando uitgevoerd (bij interactive clusters) |

**Wat te zoeken in de resultaten:**
- Welke service principal heeft de pipeline gestart? (Workflow-service principal vs. menselijke gebruiker)
- Zijn er `runFailed`-events? Dit kan wijzen op configuratieproblemen.
- Hoe vaak worden pipelines gedraaid? (Frequentie-analyse voor capacity planning)
- Zijn er onverwachte `updatePipeline`-events? (Wie heeft de DLT-definitie gewijzigd?)

In [ ]:
pipeline_events_sql = f"""
SELECT
    event_time,
    user_identity.email                         AS user_or_principal,
    action_name,
    service_name,
    request_params.job_id                       AS job_id,
    request_params.run_id                       AS run_id,
    request_params.pipeline_id                  AS pipeline_id,
    response.status_code                        AS status_code
FROM system.access.audit
WHERE event_time >= CURRENT_TIMESTAMP() - INTERVAL {lookback_days} DAYS
  AND (
        service_name IN ('jobs', 'pipelines', 'lakeflow')
     OR action_name IN (
          'runStart', 'runSucceeded', 'runFailed', 'runNow',
          'pipelineStarted', 'pipelineStopped', 'pipelineCompleted',
          'createPipeline', 'updatePipeline', 'deletePipeline',
          'createJob', 'updateJob', 'deleteJob', 'runJobNow'
        )
  )
ORDER BY event_time DESC
LIMIT 100
"""

pipeline_df = spark.sql(pipeline_events_sql)
count = pipeline_df.count()
print(f"Pipeline run events in de afgelopen {lookback_days} dag(en): {count} (max 100 getoond)")
pipeline_df.show(truncate=False)

## Stap 5 — Samenvatting: unieke gebruikers & meest actieve acties

### Wat laat dit zien?

Een geaggregeerde weergave van de activiteit op de demo-catalog: wie is het meest actief geweest, en welke acties zijn het meest uitgevoerd? Dit is een compacte weergave die geschikt is voor een **compliance-rapport** of **management-dashboard**.

**Wat te zoeken in de resultaten:**
- Zijn er gebruikers met onverwacht veel activiteit?
- Domineert één `action_name` (bijv. honderden `getTable` vs. een handvol `writeTable`)?
- Zijn er acties met `status_code != 200`? Dit wijst op gefaalde pogingen (permissiefouten, etc.).

In [ ]:
summary_sql = f"""
SELECT
    user_identity.email                         AS user_or_principal,
    action_name,
    response.status_code                        AS status_code,
    COUNT(*)                                    AS event_count,
    MIN(event_time)                             AS first_seen,
    MAX(event_time)                             AS last_seen
FROM system.access.audit
WHERE event_time >= CURRENT_TIMESTAMP() - INTERVAL {lookback_days} DAYS
  AND (
        LOWER(request_params.catalog_name) = LOWER('{catalog}')
     OR LOWER(request_params.full_name_arg) LIKE LOWER('{catalog}.%')
  )
GROUP BY
    user_identity.email,
    action_name,
    response.status_code
ORDER BY event_count DESC
LIMIT 50
"""

summary_df = spark.sql(summary_sql)
count = summary_df.count()
print(f"Unieke (gebruiker, actie, status)-combinaties: {count}")
summary_df.show(truncate=False)

## Resultaat — Compliance & Governance narratief

Dit notebook heeft de volgende audit-sporen gedemonstreerd:

| Spoor | Tabel/bron |
|---|---|
| Recente activiteit op `DEMO_DEV.*` | `system.access.audit` gefilterd op catalog-naam |
| Aanmaak/wijziging control table | `system.access.audit` gefilterd op `CONFIG.pipeline_sources` |
| Pipeline run events | `system.access.audit` gefilterd op `service_name` / `action_name` |
| Gebruikers- & actie-samenvatting | Geaggregeerde query voor compliance-rapportage |

### Waarom dit waardevol is voor klanten

1. **Geen extra tooling nodig** — de audit logs zijn een ingebouwd onderdeel van Unity Catalog, geen apart SIEM-product.
2. **Onveranderlijk spoor** — gebruikers kunnen de audit-records niet aanpassen; de integriteit is platform-gewaarborgd.
3. **Direct querybaar via SQL** — data-engineers en data-analisten kunnen zonder speciale tools compliance-queries uitvoeren.
4. **Combineerbaar met Delta Time Travel** — voor elke data-wijziging is zowel **wie** (audit log) als **wat** (Delta history) aantoonbaar.
5. **Schaalbaar** — de `system.access.audit`-tabel groeit mee met de hoeveelheid activiteit en blijft querybaar via Spark SQL.

### Volgende stappen (voor de klant)

- Stel een Databricks SQL Alert in op acties zoals `deleteTable` of gefaalde lees-pogingen.
- Exporteer de audit-query-resultaten naar een externe SIEM (bijv. Microsoft Sentinel) via een geplande Job.
- Combineer met Unity Catalog Lineage (zie `demo_showcase/lineage.ipynb`) voor een volledig data-herkomst-spoor.